In [37]:
import cv2
import numpy as np
import pytesseract
from passporteye import read_mrz
import re

def display_image(title, img):
    """ Display image at each step """
    cv2.imshow(title, img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

def deskew_image(image):
    """ Deskew image using Hough Line Transform """
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150, apertureSize=3)
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, 100, minLineLength=100, maxLineGap=10)

    if lines is not None:
        angles = [np.arctan2(y2 - y1, x2 - x1) for [[x1, y1, x2, y2]] in lines]
        median_angle = np.median(angles)
        (h, w) = image.shape[:2]
        center = (w // 2, h // 2)
        M = cv2.getRotationMatrix2D(center, np.degrees(median_angle), 1.0)
        image = cv2.warpAffine(image, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)

    return image

def preprocess_image(image):
    """ Apply adaptive thresholding and morphological transformations """
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    gray = cv2.bilateralFilter(gray, 9, 75, 75)  # Reduce noise while preserving edges

    # Adaptive Thresholding
    adaptive_thresh = cv2.adaptiveThreshold(
        gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 2
    )

    # Morphological Operations
    kernel = np.ones((1, 1), np.uint8)
    processed = cv2.morphologyEx(adaptive_thresh, cv2.MORPH_CLOSE, kernel)

    return processed

def extract_mrz(image_path):
    """ Extract MRZ from passport using PassportEye """
    mrz = read_mrz(image_path)
    if mrz:
        return mrz.to_dict()
    return None

def format_date(mrz_date):
    """ Convert MRZ date format YYMMDD to DD/MM/YY """
    if len(mrz_date) == 6:
        return f"{mrz_date[4:6]}/{mrz_date[2:4]}/{mrz_date[0:2]}"
    return mrz_date

def clean_mrz_text(text):
    """ Remove unwanted < and extra characters from MRZ output """
    text = text.replace("<", " ").strip()
    text = re.sub(r"\s+", " ", text)  # Remove extra spaces
    return text

def format_mrz_data(mrz_data):
    """ Format MRZ extracted data properly """
    first_name = clean_mrz_text(mrz_data.get("surname", ""))
    surname = clean_mrz_text(mrz_data.get("names", ""))

    # Remove trailing "K<K<KKKK..." issue from Surname
    surname = re.sub(r'K+\s*K+', '', surname).strip()

    formatted_data = {
        "First Name": first_name,  # Swapped Surname & First Name
        "Surname": surname,  # Swapped First Name & Surname
        "Passport Number": mrz_data.get("number", "").replace("<", ""),
        "Date of Birth": format_date(mrz_data.get("date_of_birth", "")),
        "Gender": "Male" if mrz_data.get("sex", "") == "M" else "Female",
        "Nationality": mrz_data.get("nationality", "").replace("<", ""),
        "Expiration Date": format_date(mrz_data.get("expiration_date", "")),
        "MRZ Raw Text": mrz_data.get("raw_text", ""),
    }
    return formatted_data

def process_passport(image_path):
    """ Main pipeline for passport OCR """
    image = cv2.imread(image_path)

    if image is None:
        print(f"Error: Unable to load image. Check file path: {image_path}")
        return

    image = deskew_image(image)
    display_image("Deskewed Image", image)

    processed = preprocess_image(image)
    display_image("Preprocessed Image", processed)

    mrz_data = extract_mrz(image_path)  # Use file path for MRZ extraction

    if mrz_data:
        formatted_mrz = format_mrz_data(mrz_data)

        print("\n--- Extracted Passport Details ---")
        for key, value in formatted_mrz.items():
            if key != "MRZ Raw Text":  # Avoid printing raw text here
                print(f"{key}: {value}")

        print("\n--- MRZ Raw Text ---")
        print(mrz_data["raw_text"])
    else:
        print("\nNo MRZ Data Found.")

if __name__ == "__main__":
    image_path = "passport/32.jpg"  # Change to correct path
    process_passport(image_path)



c:\Users\tarun.pithani\AppData\Local\Programs\Python\Python313\Lib\site-packages\passporteye\mrz\image.py:37: FutureWarning: The plugin infrastructure in `skimage.io` and the parameter `plugin` are deprecated since version 0.25 and will be removed in 0.27 (or later). To avoid this warning, please do not use the parameter `plugin`. Instead, use `imageio` or other I/O packages directly. See also `imread`.
  img = skimage_io.imread(file, as_gray=self.as_gray, plugin='imageio')
c:\Users\tarun.pithani\AppData\Local\Programs\Python\Python313\Lib\site-packages\passporteye\mrz\image.py:89: FutureWarning: `square` is deprecated since version 0.25 and will be removed in version 0.27. Use `skimage.morphology.footprint_rectangle` instead.
  m = morphology.square(self.square_size)



--- Extracted Passport Details ---
First Name: SPECIMEN
Surname: EMMA
Passport Number: TADO0000
Date of Birth: 25/12/84
Gender: Female
Nationality: XXB
Expiration Date: 27/03/19

--- MRZ Raw Text ---
P<BELSPECIMEN<<EMMA<<<<<<<<<<<<KKKKKKKKKKKKK
TADO0000<3XXB8412254F1903278<<<<<<<<<<<<<<04
